# Alzheimer MRI Classification
**Model :** EfficientNet-B0 (fine-tuned, ImageNet weights)  
**Classes :** NonDemented · VeryMildDemented · MildDemented · ModerateDemented  
**Primary metric :** Macro F1 *(test set imbalanced — Moderate class has very few samples)*  
**Epochs :** 17 (with early stopping on val Macro F1)

##  Setup imports

In [ ]:
import sys, os
from pathlib import Path
sys.path.append(os.path.abspath('src'))   # make src/ importable

import torch
import random
import numpy as np
import matplotlib.pyplot as plt
import torchvision

# ── project modules ──
from data.data_loader_alzheimer  import get_dataloaders, denormalize
from model.model_alzheimer       import build_efficientnet_b0, count_parameters
from model.train_utils_alzheimer import train_model, evaluate
from config import get_config, get_paths
from utils.output_manager import get_output_manager

# ── reproducibility ──
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU   : {torch.cuda.get_device_name(0)}')

Device: cpu


## 1. Configuration

In [ ]:


cfg = get_config('Alzheimer')
paths = get_paths('Alzheimer')
manager = get_output_manager('Alzheimer')

DATA_ROOT = str(Path(cfg['paths']['data_root']).resolve())
RESULTS_DIR = str(paths['disease_root'])
WEIGHTS_PATH = str(manager.paths['weights'] / Path(cfg['paths']['weights']).name)

IMG_SIZE     = cfg['data']['img_size']
BATCH_SIZE   = cfg['data']['batch_size']
NUM_WORKERS  = cfg['data'].get('num_workers', 2)
EPOCHS       = cfg['model']['epochs']
LR           = cfg['model']['learning_rate']
WEIGHT_DECAY = cfg['model']['weight_decay']
PATIENCE     = cfg['model']['patience']

manager.print_structure()
print('✅ Config loaded from', paths['config'])

✅ Config ready.


## 2. Data Loading

In [20]:
train_loader, val_loader, test_loader, CLASS_NAMES = get_dataloaders(
    data_root   = DATA_ROOT,
    img_size    = IMG_SIZE,
    batch_size  = BATCH_SIZE,
    num_workers = NUM_WORKERS,
    seed        = SEED,
)
NUM_CLASSES = len(CLASS_NAMES)
print(f'Classes ({NUM_CLASSES}): {CLASS_NAMES}')

[AlzheimerLoader] Train=8192 | Val=2048 | Test=1279
[AlzheimerLoader] Classes: ['Mild Impairment', 'Moderate Impairment', 'No Impairment', 'Very Mild Impairment']
Classes (4): ['Mild Impairment', 'Moderate Impairment', 'No Impairment', 'Very Mild Impairment']


In [ ]:
# Visualise one augmented batch
imgs, labels = next(iter(train_loader))
grid = torchvision.utils.make_grid(denormalize(imgs[:16]), nrow=8)
plt.figure(figsize=(16, 4))
plt.imshow(grid.permute(1, 2, 0))
plt.title('Augmented Training Batch — Alzheimer', fontsize=13)
plt.axis('off')
plt.tight_layout()
plt.savefig(str(manager.paths['plots'] / 'sample_batch_alzheimer.png'), dpi=150)
plt.show()

C:\Users\nour\AppData\Local\Temp\ipykernel_14196\2158388923.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Model

In [22]:
model = build_efficientnet_b0(num_classes=NUM_CLASSES)
print(f'EfficientNet-B0 | trainable params: {count_parameters(model):.2f}M')

EfficientNet-B0 | trainable params: 4.01M


## 4. Training  *(17 epochs, early stopping on val Macro F1)*

In [23]:
history, model = train_model(
    model        = model,
    train_loader = train_loader,
    val_loader   = val_loader,
    device       = DEVICE,
    epochs       = EPOCHS,
    lr           = LR,
    weight_decay = WEIGHT_DECAY,
    patience     = PATIENCE,
)

# Save weights
torch.save(model.state_dict(), WEIGHTS_PATH)
print(f'\n Best weights saved → {WEIGHTS_PATH}')

  Ep 01/17 | tr_loss=0.9466 tr_acc=0.6434 | val_loss=0.7275 val_acc=0.7725 val_macroF1=0.7712 ✅ best


c:\Users\nour\miniconda3\envs\cnn\lib\site-packages\torch\utils\data\dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


  Ep 02/17 | tr_loss=0.7129 tr_acc=0.7959 | val_loss=0.6690 val_acc=0.8115 val_macroF1=0.8073 ✅ best


c:\Users\nour\miniconda3\envs\cnn\lib\site-packages\torch\utils\data\dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


  Ep 03/17 | tr_loss=0.6576 tr_acc=0.8229 | val_loss=0.6403 val_acc=0.8286 val_macroF1=0.8287 ✅ best


...

  Ep 15/17 | tr_loss=0.4111 tr_acc=0.9760 | val_loss=0.4254 val_acc=0.9663 val_macroF1=0.9664


c:\Users\nour\miniconda3\envs\cnn\lib\site-packages\torch\utils\data\dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


  Ep 16/17 | tr_loss=0.4103 tr_acc=0.9771 | val_loss=0.4213 val_acc=0.9673 val_macroF1=0.9673


c:\Users\nour\miniconda3\envs\cnn\lib\site-packages\torch\utils\data\dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


  Ep 17/17 | tr_loss=0.4133 tr_acc=0.9733 | val_loss=0.4241 val_acc=0.9673 val_macroF1=0.9674
  ⏱️  Done in 457.1 min

 Best weights saved → ./models/efficientnet_b0_alzheimer.pt


## 5. Evaluation on Test Set

In [24]:
import torch.nn as nn
criterion = nn.CrossEntropyLoss()

loss, acc, preds, labels, probs, macro_f1 = evaluate(
    model, test_loader, criterion, DEVICE
)
print(f'Test Loss     : {loss:.4f}')
print(f'Test Accuracy : {acc:.4f}')
print(f'Test Macro F1 : {macro_f1:.4f}  ← PRIMARY METRIC')

c:\Users\nour\miniconda3\envs\cnn\lib\site-packages\torch\utils\data\dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Test Loss     : 0.2459
Test Accuracy : 0.9304
Test Macro F1 : 0.9459  ← PRIMARY METRIC


## 6. Save All Results & Plots

In [ ]:
manager.save_training_history(history)
manager.save_training_curves(history, 'Alzheimer MRI — EfficientNet-B0')
manager.save_confusion_matrix(labels, preds, CLASS_NAMES,
                              title='Alzheimer Confusion Matrix')
manager.save_metrics_report({
    'loss': float(loss),
    'accuracy': float(acc),
    'macro_f1': float(macro_f1),
    'classes': CLASS_NAMES,
})

print('\n Results saved:')
for fn in sorted(os.listdir(RESULTS_DIR)):
    print(f'   {fn}')

[saved] ./results/alzheimer\learning_curves_alzheimer.png
[saved] ./results/alzheimer\confusion_matrix_alzheimer.png
[saved] ./results/alzheimer\roc_curves_alzheimer.png
[saved] ./results/alzheimer\results_alzheimer.json

  Macro F1    : 0.9459  ← PRIMARY
  Weighted F1 : 0.9308
  ROC-AUC     : 0.9903
  Accuracy    : 0.9304

⚠️  NOTE: ModerateDemented has very few test samples.
   Per-class metrics for this class are indicative only.

                      precision    recall  f1-score   support

     Mild Impairment     0.9583    0.8994    0.9280       179
 Moderate Impairment     1.0000    1.0000    1.0000        12
       No Impairment     0.9610    0.9250    0.9427       640
Very Mild Impairment     0.8799    0.9487    0.9130       448

            accuracy                         0.9304      1279
           macro avg     0.9498    0.9433    0.9459      1279
        weighted avg     0.9326    0.9304    0.9308      1279


 Results saved:
   confusion_matrix_alzheimer.png
   learning_

## 7. Display Saved Plots

In [26]:
from PIL import Image as PILImage

plots = [
    ('Learning Curves',   'learning_curves_alzheimer.png'),
    ('Confusion Matrix',  'confusion_matrix_alzheimer.png'),
    ('ROC Curves',        'roc_curves_alzheimer.png'),
]

fig, axes = plt.subplots(1, 3, figsize=(21, 6))
for ax, (title, fname) in zip(axes, plots):
    img = PILImage.open(os.path.join(RESULTS_DIR, fname))
    ax.imshow(img)
    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.axis('off')
plt.tight_layout()
plt.show()

C:\Users\nour\AppData\Local\Temp\ipykernel_14196\3923055139.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
